# Operational / Descriptive Statistics — Olist

Notebook này bổ sung một nhánh **descriptive / operational analytics** cho bài toán Customer Churn.

Mục tiêu:
- Đặt câu hỏi thực tế từ dữ liệu e-commerce.
- Tính KPI bằng PySpark.
- Chuyển các bảng summary nhỏ sang Pandas để trực quan hóa bằng hvPlot/Bokeh.
- Không thay đổi RFM, K-Means, Churn Labeling hay mô hình ML.

**Grain sử dụng theo từng bài toán:**
- Order Status: 1 dòng = 1 order.
- Delivery: 1 dòng = 1 delivered order.
- Review: 1 dòng = 1 review_id.
- Repurchase: 1 dòng = 1 customer.

In [1]:
%pip -q install hvplot panel

from pathlib import Path
import pandas as pd
import hvplot.pandas
import panel as pn

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

pn.extension()

DATA_DIR = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")
PROCESSED_DIR = DATA_DIR / "processed"

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu: {DATA_DIR}")

spark = (
    SparkSession.builder
    .appName("Olist-Operational-Descriptive-Statistics")
    .getOrCreate()
)

def read_csv(path, multiline=False):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    return (
        spark.read
        .option("header", True)
        .option("multiLine", multiline)
        .option("escape", '"')
        .option("inferSchema", True)
        .csv(str(path))
    )

print("DATA_DIR:", DATA_DIR)
print("Spark:", spark.version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 4.3 MB/s eta 0:00:00


/tmp/ipykernel_1346/13144678.py:11: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/BigData/DA-BigData/code/data
Spark: 4.0.4


In [2]:
orders = read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = read_csv(DATA_DIR / "olist_customers_dataset.csv")
items = read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = read_csv(DATA_DIR / "olist_order_reviews_dataset.csv", multiline=True)

print("Orders   :", f"{orders.count():,}")
print("Customers:", f"{customers.count():,}")
print("Items    :", f"{items.count():,}")
print("Payments :", f"{payments.count():,}")
print("Reviews  :", f"{reviews.count():,}")

Orders   : 99,441
Customers: 99,441
Items    : 112,650
Payments : 103,886
Reviews  : 99,224


## 1. Order Status & Cancellation

**Câu hỏi thực tế:** Trong toàn bộ đơn hàng, có bao nhiêu đơn bị hủy và tỷ lệ hủy là bao nhiêu?

Công thức:

\[
Cancellation\ Rate =
\frac{Canceled\ Orders}{Total\ Orders} \times 100\%
\]

Phân tích này mô tả tình trạng vận hành, chưa kết luận nguyên nhân của churn.

In [3]:
order_status_summary = (
    orders
    .groupBy("order_status")
    .agg(F.count("*").alias("order_count"))
    .withColumn(
        "order_rate_pct",
        F.round(
            F.col("order_count") * 100 / F.sum("order_count").over(Window.partitionBy()),
            2
        )
    )
    .orderBy(F.desc("order_count"))
)

order_status_summary.show(truncate=False)

cancellation_summary = (
    orders
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("order_status") == "canceled", 1).otherwise(0)).alias("canceled_orders"),
        F.sum(F.when(F.col("order_status") == "unavailable", 1).otherwise(0)).alias("unavailable_orders")
    )
    .withColumn(
        "cancellation_rate_pct",
        F.round(F.col("canceled_orders") * 100 / F.col("total_orders"), 2)
    )
)

cancellation_summary.show(truncate=False)

+------------+-----------+--------------+
|order_status|order_count|order_rate_pct|
+------------+-----------+--------------+
|delivered   |96478      |97.02         |
|shipped     |1107       |1.11          |
|canceled    |625        |0.63          |
|unavailable |609        |0.61          |
|invoiced    |314        |0.32          |
|processing  |301        |0.3           |
|created     |5          |0.01          |
|approved    |2          |0.0           |
+------------+-----------+--------------+

+------------+---------------+------------------+---------------------+
|total_orders|canceled_orders|unavailable_orders|cancellation_rate_pct|
+------------+---------------+------------------+---------------------+
|99441       |625            |609               |0.63                 |
+------------+---------------+------------------+---------------------+



In [4]:
status_pd = order_status_summary.toPandas()

status_chart = status_pd.hvplot.bar(
    x="order_status",
    y="order_count",
    width=850,
    height=420,
    rot=35,
    title="Order Status Distribution",
    xlabel="Order status",
    ylabel="Number of orders",
    tools=["hover"]
)

status_chart

:Bars   [order_status]   (order_count)

## 2. Cancellation Trend theo tháng

**Câu hỏi:** Tỷ lệ đơn bị hủy có thay đổi theo thời gian hay không?

Tỷ lệ được tính trên **tất cả orders phát sinh trong từng tháng**, không chỉ trên các đơn delivered.

In [5]:
monthly_cancellation = (
    orders
    .withColumn("purchase_at", F.to_timestamp("order_purchase_timestamp"))
    .filter(F.col("purchase_at").isNotNull())
    .withColumn("purchase_month", F.trunc(F.to_date("purchase_at"), "month"))
    .groupBy("purchase_month")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("order_status") == "canceled", 1).otherwise(0)).alias("canceled_orders")
    )
    .withColumn(
        "cancellation_rate_pct",
        F.round(F.col("canceled_orders") * 100 / F.col("total_orders"), 2)
    )
    .orderBy("purchase_month")
)

monthly_cancellation.show(100, truncate=False)

cancel_pd = monthly_cancellation.toPandas()
cancel_pd["purchase_month"] = pd.to_datetime(cancel_pd["purchase_month"])

cancel_chart = cancel_pd.hvplot.line(
    x="purchase_month",
    y="cancellation_rate_pct",
    width=1100,
    height=420,
    marker="o",
    title="Monthly Cancellation Rate",
    xlabel="Purchase month",
    ylabel="Cancellation rate (%)",
    tools=["hover"]
)

cancel_chart

+--------------+------------+---------------+---------------------+
|purchase_month|total_orders|canceled_orders|cancellation_rate_pct|
+--------------+------------+---------------+---------------------+
|2016-09-01    |4           |2              |50.0                 |
|2016-10-01    |324         |24             |7.41                 |
|2016-12-01    |1           |0              |0.0                  |
|2017-01-01    |800         |3              |0.38                 |
|2017-02-01    |1780        |17             |0.96                 |
|2017-03-01    |2682        |33             |1.23                 |
|2017-04-01    |2404        |18             |0.75                 |
|2017-05-01    |3700        |29             |0.78                 |
|2017-06-01    |3245        |16             |0.49                 |
|2017-07-01    |4026        |28             |0.7                  |
|2017-08-01    |4331        |27             |0.62                 |
|2017-09-01    |4285        |20             |0.4

:Curve   [purchase_month]   (cancellation_rate_pct)

## 3. Delivery Performance

**Câu hỏi thực tế:**
- Bao nhiêu đơn giao trễ?
- Khách hàng phải chờ bao lâu?
- Đuôi phân phối giao hàng dài đến mức nào?

Chỉ dùng các đơn có đủ mốc:
`purchase → carrier → delivered`.

Đơn được xem là **late** khi ngày giao thực tế lớn hơn ngày giao dự kiến.

In [6]:
delivery = (
    orders
    .filter(F.col("order_status") == "delivered")
    .withColumn("purchase_at", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("carrier_at", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("delivered_at", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("estimated_on", F.to_date("order_estimated_delivery_date"))
    .withColumn(
        "valid_delivery",
        F.col("purchase_at").isNotNull()
        & F.col("carrier_at").isNotNull()
        & F.col("delivered_at").isNotNull()
        & (F.col("carrier_at") >= F.col("purchase_at"))
        & (F.col("delivered_at") >= F.col("carrier_at"))
    )
    .withColumn(
        "delivery_days",
        F.when(
            F.col("valid_delivery"),
            (
                F.unix_timestamp("delivered_at")
                - F.unix_timestamp("purchase_at")
            ) / F.lit(86400.0)
        )
    )
    .withColumn(
        "late_flag",
        F.when(
            F.col("valid_delivery") & F.col("estimated_on").isNotNull(),
            F.when(F.to_date("delivered_at") > F.col("estimated_on"), 1).otherwise(0)
        )
    )
)

delivery_kpi = (
    delivery
    .agg(
        F.count("*").alias("delivered_orders"),
        F.count("delivery_days").alias("valid_delivery_orders"),
        F.count("late_flag").alias("orders_eligible_for_late_check"),
        F.sum(F.coalesce(F.col("late_flag"), F.lit(0))).alias("late_orders"),
        F.percentile_approx("delivery_days", [0.50, 0.75, 0.90, 0.95], 10000).alias("delivery_quantiles")
    )
    .select(
        "delivered_orders",
        "valid_delivery_orders",
        "orders_eligible_for_late_check",
        "late_orders",
        F.round(
            F.col("late_orders") * 100 / F.col("orders_eligible_for_late_check"), 2
        ).alias("late_delivery_rate_pct"),
        F.round(F.col("delivery_quantiles")[0], 2).alias("p50_delivery_days"),
        F.round(F.col("delivery_quantiles")[1], 2).alias("p75_delivery_days"),
        F.round(F.col("delivery_quantiles")[2], 2).alias("p90_delivery_days"),
        F.round(F.col("delivery_quantiles")[3], 2).alias("p95_delivery_days")
    )
)

delivery_kpi.show(truncate=False)

+----------------+---------------------+------------------------------+-----------+----------------------+-----------------+-----------------+-----------------+-----------------+
|delivered_orders|valid_delivery_orders|orders_eligible_for_late_check|late_orders|late_delivery_rate_pct|p50_delivery_days|p75_delivery_days|p90_delivery_days|p95_delivery_days|
+----------------+---------------------+------------------------------+-----------+----------------------+-----------------+-----------------+-----------------+-----------------+
|96478           |96281                |96281                         |6531       |6.78                  |10.23            |15.73            |23.1             |29.27            |
+----------------+---------------------+------------------------------+-----------+----------------------+-----------------+-----------------+-----------------+-----------------+



In [7]:
delivery_dist = (
    delivery
    .filter(F.col("delivery_days").isNotNull())
    .select("order_id", "delivery_days")
)

delivery_pd = delivery_dist.toPandas()

delivery_chart = delivery_pd.hvplot.hist(
    y="delivery_days",
    bins=50,
    width=950,
    height=420,
    title="Distribution of Delivery Days",
    xlabel="Delivery days",
    ylabel="Number of orders"
)

delivery_chart

:Histogram   [delivery_days]   (Count)

## 4. Review Distribution

**Câu hỏi:** Chất lượng phản hồi của khách hàng được phân bố như thế nào?

Review score được lấy ở mức `review_id`.  
Nhóm negative review được định nghĩa mô tả là **1–2 sao**.

In [8]:
review_score = (
    reviews
    .filter(F.col("review_id").isNotNull())
    .withColumn("score", F.col("review_score").cast("int"))
    .filter(F.col("score").between(1, 5))
    .groupBy("score")
    .agg(F.count("*").alias("review_count"))
    .orderBy("score")
)

review_total = review_score.agg(F.sum("review_count").alias("total")).first()["total"]

review_distribution = (
    review_score
    .withColumn(
        "review_rate_pct",
        F.round(F.col("review_count") * 100 / F.lit(review_total), 2)
    )
)

review_distribution.show(truncate=False)

negative_review = (
    review_distribution
    .filter(F.col("score").isin(1, 2))
    .agg(F.sum("review_count").alias("negative_reviews"))
    .withColumn(
        "negative_review_rate_pct",
        F.round(F.col("negative_reviews") * 100 / F.lit(review_total), 2)
    )
)

negative_review.show(truncate=False)

+-----+------------+---------------+
|score|review_count|review_rate_pct|
+-----+------------+---------------+
|1    |11424       |11.51          |
|2    |3151        |3.18           |
|3    |8179        |8.24           |
|4    |19142       |19.29          |
|5    |57328       |57.78          |
+-----+------------+---------------+

+----------------+------------------------+
|negative_reviews|negative_review_rate_pct|
+----------------+------------------------+
|14575           |14.69                   |
+----------------+------------------------+



In [9]:
review_pd = review_distribution.toPandas()

review_chart = review_pd.hvplot.bar(
    x="score",
    y="review_rate_pct",
    width=750,
    height=420,
    title="Review Score Distribution",
    xlabel="Review score",
    ylabel="Share of reviews (%)",
    tools=["hover"]
)

review_chart

ValueError: Changing the dtype of a 0d array is only supported if the itemsize is unchanged

:Bars   [score]   (review_rate_pct)

## 5. One-time Buyer vs Repeat Buyer

**Câu hỏi thực tế:** Khách hàng của Olist chủ yếu mua một lần hay quay lại mua?

Chỉ dùng các orders có `order_status = delivered`.

Một khách:
- `1 purchase` → one-time buyer
- `>=2 purchases` → repeat buyer

In [10]:
delivered_customer_orders = (
    orders
    .filter(F.col("order_status") == "delivered")
    .select("order_id", "customer_id")
    .join(
        customers.select("customer_id", "customer_unique_id"),
        "customer_id",
        "inner"
    )
)

customer_purchase_count = (
    delivered_customer_orders
    .groupBy("customer_unique_id")
    .agg(F.countDistinct("order_id").alias("purchase_count"))
)

purchase_group = (
    customer_purchase_count
    .withColumn(
        "customer_type",
        F.when(F.col("purchase_count") == 1, "One-time buyer")
        .otherwise("Repeat buyer")
    )
)

repurchase_summary = (
    purchase_group
    .groupBy("customer_type")
    .agg(F.count("*").alias("customer_count"))
)

total_customers = purchase_group.count()

repurchase_summary = (
    repurchase_summary
    .withColumn(
        "customer_rate_pct",
        F.round(F.col("customer_count") * 100 / F.lit(total_customers), 2)
    )
    .orderBy(F.desc("customer_count"))
)

repurchase_summary.show(truncate=False)

+--------------+--------------+-----------------+
|customer_type |customer_count|customer_rate_pct|
+--------------+--------------+-----------------+
|One-time buyer|90557         |97.0             |
|Repeat buyer  |2801          |3.0              |
+--------------+--------------+-----------------+



In [11]:
repurchase_pd = repurchase_summary.toPandas()

repurchase_chart = repurchase_pd.hvplot.bar(
    x="customer_type",
    y="customer_rate_pct",
    width=800,
    height=420,
    title="One-time vs Repeat Buyers",
    xlabel="Customer type",
    ylabel="Share of customers (%)",
    tools=["hover"]
)

repurchase_chart

:Bars   [customer_type]   (customer_rate_pct)

## 6. Order Value & Freight

**Câu hỏi:** Một đơn hàng có giá trị bao nhiêu và freight chiếm tỷ trọng thế nào?

Tính ở mức `order_id`:
- `order_value` = tổng `price`
- `freight_total` = tổng `freight_value`
- `freight_ratio` = freight / (price + freight)

Chỉ dùng delivered orders và loại các giá trị không hợp lệ.

In [12]:
delivered_ids = orders.filter(F.col("order_status") == "delivered").select("order_id").dropDuplicates()

item_order_value = (
    items
    .join(delivered_ids, "order_id", "inner")
    .withColumn("price_num", F.col("price").cast("double"))
    .withColumn("freight_num", F.col("freight_value").cast("double"))
    .groupBy("order_id")
    .agg(
        F.sum("price_num").alias("order_value"),
        F.sum("freight_num").alias("freight_total"),
        F.count("*").alias("item_count")
    )
    .withColumn(
        "freight_ratio",
        F.when(
            (F.col("order_value") + F.col("freight_total")) > 0,
            F.col("freight_total") /
            (F.col("order_value") + F.col("freight_total"))
        )
    )
)

value_kpi = (
    item_order_value
    .agg(
        F.count("*").alias("orders"),
        F.percentile_approx(
            "order_value", [0.50, 0.75, 0.90, 0.95], 10000
        ).alias("order_value_q"),
        F.percentile_approx(
            "freight_total", [0.50, 0.75, 0.90, 0.95], 10000
        ).alias("freight_q"),
        F.percentile_approx(
            "freight_ratio", [0.50, 0.75, 0.90, 0.95], 10000
        ).alias("freight_ratio_q")
    )
    .select(
        "orders",
        F.round(F.col("order_value_q")[0], 2).alias("order_value_p50"),
        F.round(F.col("order_value_q")[1], 2).alias("order_value_p75"),
        F.round(F.col("order_value_q")[2], 2).alias("order_value_p90"),
        F.round(F.col("order_value_q")[3], 2).alias("order_value_p95"),
        F.round(F.col("freight_q")[0], 2).alias("freight_p50"),
        F.round(F.col("freight_q")[1], 2).alias("freight_p75"),
        F.round(F.col("freight_q")[2], 2).alias("freight_p90"),
        F.round(F.col("freight_ratio_q")[0] * 100, 2).alias("freight_ratio_p50_pct"),
        F.round(F.col("freight_ratio_q")[1] * 100, 2).alias("freight_ratio_p75_pct"),
        F.round(F.col("freight_ratio_q")[2] * 100, 2).alias("freight_ratio_p90_pct"),
        F.round(F.col("freight_ratio_q")[3] * 100, 2).alias("freight_ratio_p95_pct")
    )
)

value_kpi.show(truncate=False)

+------+---------------+---------------+---------------+---------------+-----------+-----------+-----------+---------------------+---------------------+---------------------+---------------------+
|orders|order_value_p50|order_value_p75|order_value_p90|order_value_p95|freight_p50|freight_p75|freight_p90|freight_ratio_p50_pct|freight_ratio_p75_pct|freight_ratio_p90_pct|freight_ratio_p95_pct|
+------+---------------+---------------+---------------+---------------+-----------+-----------+-----------+---------------------+---------------------+---------------------+---------------------+
|96478 |86.5           |149.9          |269.0          |399.0          |17.17      |24.01      |39.35      |18.33                |27.56                |38.35                |45.59                |
+------+---------------+---------------+---------------+---------------+-----------+-----------+-----------+---------------------+---------------------+---------------------+---------------------+



## 7. Customer Journey / Funnel

**Câu hỏi:** Từ một khách hàng đã mua hàng, bao nhiêu khách:
1. có order delivered,
2. có review,
3. quay lại mua ít nhất lần hai?

Đây là descriptive funnel, không phải causal analysis.

In [13]:
customer_base = (
    delivered_customer_orders
    .select("customer_unique_id")
    .distinct()
)

review_customers = (
    reviews
    .filter(F.col("review_id").isNotNull())
    .select("order_id")
    .distinct()
    .join(
        delivered_customer_orders.select("order_id", "customer_unique_id"),
        "order_id",
        "inner"
    )
    .select("customer_unique_id")
    .distinct()
)

repeat_customers = (
    purchase_group
    .filter(F.col("purchase_count") >= 2)
    .select("customer_unique_id")
    .distinct()
)

funnel = pd.DataFrame({
    "stage": [
        "Delivered customer",
        "Customer with review",
        "Repeat buyer"
    ],
    "customer_count": [
        customer_base.count(),
        review_customers.count(),
        repeat_customers.count()
    ]
})

funnel["rate_vs_delivered_customer_pct"] = (
    funnel["customer_count"] * 100 / funnel.loc[0, "customer_count"]
).round(2)

display(funnel)

funnel_chart = funnel.hvplot.bar(
    x="stage",
    y="rate_vs_delivered_customer_pct",
    width=900,
    height=420,
    rot=20,
    title="Customer Journey Funnel",
    xlabel="Journey stage",
    ylabel="Share of delivered customers (%)",
    tools=["hover"]
)

funnel_chart

,stage,customer_count,rate_vs_delivered_customer_pct
0,Delivered customer,93358,100.00
1,Customer with review,92755,99.35
2,Repeat buyer,2801,3.00


:Bars   [stage]   (rate_vs_delivered_customer_pct)

## 8. Delivery Outcome ↔ Churn 180

Phần này là **retrospective descriptive analysis**.

Mục tiêu không phải chứng minh giao trễ gây churn, mà chỉ hỏi:

> Trong dữ liệu đã có `churn_180`, tỷ lệ churn quan sát được khác nhau thế nào giữa các order có delivery đúng hạn và giao trễ?

Phần này chỉ chạy nếu file model-input hiện tại tồn tại và có `churn_180` cùng `snapshot_order_id`.

In [14]:
MODEL_INPUT = DATA_DIR / "model_input" / "customer_churn_model_dataset.csv"

if MODEL_INPUT.exists():
    churn_data = (
        read_csv(MODEL_INPUT)
        .select("snapshot_order_id", "churn_180")
        .withColumn("churn_180", F.col("churn_180").cast("int"))
    )

    delivery_outcome = (
        delivery
        .select(
            "order_id",
            F.when(F.col("late_flag") == 1, "Late")
             .when(F.col("late_flag") == 0, "On-time")
             .otherwise("Unknown")
             .alias("delivery_outcome")
        )
    )

    churn_delivery = (
        churn_data
        .join(
            delivery_outcome,
            churn_data.snapshot_order_id == delivery_outcome.order_id,
            "inner"
        )
        .filter(F.col("delivery_outcome").isin("Late", "On-time"))
        .groupBy("delivery_outcome")
        .agg(
            F.count("*").alias("observations"),
            F.avg("churn_180").alias("churn_rate")
        )
        .withColumn("churn_rate_pct", F.round(F.col("churn_rate") * 100, 2))
        .orderBy("delivery_outcome")
    )

    churn_delivery.show(truncate=False)
else:
    print("Không tìm thấy model input:", MODEL_INPUT)
    print("Bỏ qua phần retrospective Delivery ↔ churn.")

{"ts": "2026-10-01 16:14:13.804", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `snapshot_order_id` cannot be resolved. Did you mean one of the following? [`snapshot_date`, `frequency_orders`, `avg_freight`, `payment_total`, `churn_180`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor74.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o797.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `snapshot_order_id` cannot be resolved. Did you mean one of the following? [`snapshot_date`, `frequency_orders`, `avg_freight`, `payment_total`, `churn_180`]. SQLSTATE: 42703;\n'Project ['snapshot_order_id, churn_180#757]\n+- Relation

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `snapshot_order_id` cannot be resolved. Did you mean one of the following? [`snapshot_date`, `frequency_orders`, `avg_freight`, `payment_total`, `churn_180`]. SQLSTATE: 42703;
'Project ['snapshot_order_id, churn_180#757]
+- Relation [customer_unique_id#722,snapshot_date#723,followup_end_180#724,temporal_split#725,recency_days#726,frequency_orders#727,orders_last_30d#728,orders_last_90d#729,orders_last_180d#730,monetary_matched_total#731,mean_matched_order_value#732,total_items#733,sum_distinct_sellers_per_order#734,items_missing_category#735,mean_order_product_weight_g#736,credit_card_orders#737,orders_payment_not_matched#738,review_count#739,avg_review_score#740,negative_review_count#741,negative_review_ratio#742,avg_delivery_days#743,late_delivery_count#744,delivery_order_count#745,late_delivery_ratio#746,... 11 more fields] csv


## 9. Operational KPI Summary

Bảng cuối gom các KPI chính để dễ đưa sang báo cáo.

**Cách diễn giải:** đây là bằng chứng mô tả về tình trạng vận hành và hành vi khách hàng; không suy ra quan hệ nhân quả từ các KPI này.

In [ ]:
print("============================================================")
print("OPERATIONAL KPI SUMMARY")
print("============================================================")

print("\n1. ORDER STATUS")
order_status_summary.show(truncate=False)

print("\n2. CANCELLATION")
cancellation_summary.show(truncate=False)

print("\n3. DELIVERY")
delivery_kpi.show(truncate=False)

print("\n4. REVIEW")
review_distribution.show(truncate=False)
negative_review.show(truncate=False)

print("\n5. REPURCHASE")
repurchase_summary.show(truncate=False)

print("\n6. ORDER VALUE / FREIGHT")
value_kpi.show(truncate=False)

## 10. Take Notes

Sau khi chạy notebook, ghi kết quả theo cấu trúc:

### Operational findings
- Order status nào chiếm tỷ trọng lớn nhất?
- Cancellation rate là bao nhiêu?
- Late delivery rate là bao nhiêu?
- Median và P90 delivery time là bao nhiêu?
- Negative review rate là bao nhiêu?
- One-time buyer chiếm bao nhiêu?
- Freight ratio ở P50/P90 là bao nhiêu?

### Business questions
- Có vấn đề vận hành nào nổi bật cần theo dõi?
- Customer repurchase có thấp không?
- Delivery performance có đáng được đưa sang phân tích churn tiếp theo không?

### Connection to churn
- Nếu Delivery ↔ churn được chạy: tỷ lệ churn quan sát giữa Late và On-time khác nhau thế nào?
- Đây chỉ là association/descriptive evidence; không kết luận delivery delay là nguyên nhân gây churn.

### Connection to the existing pipeline

```text
Raw Olist
   ↓
Operational / Descriptive Statistics
   ↓
Spearman
   ↓
Mann–Whitney
   ↓
Kruskal–Wallis / Cohort / Survival (nếu cần)
   ↓
Feature Engineering
   ↓
K-Means / Churn Labeling
   ↓
SMOTE / Model
   ↓
XAI
```